# Shop pipeline in PySpark
Bronze, silver, gold and quality checks, same logic as the dbt project. Reads Postgres over JDBC, everything stays in Spark.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("shop")
    # Postgres JDBC driver, downloaded on first start
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4")
    .getOrCreate()
)

url = "jdbc:postgresql://localhost:5433/mydb"
props = {"user": "postgres", "password": "postgres", "driver": "org.postgresql.Driver"}

## Bronze
Raw copy of the source tables.

In [ ]:
bronze_customers = spark.read.jdbc(url, "shop.customers", properties=props)
bronze_orders = spark.read.jdbc(url, "shop.orders", properties=props)

bronze_orders.show()

## Silver
Trim names, cast amount, drop orders without an amount.

In [ ]:
silver_customers = bronze_customers.select(
    "customer_id",
    # remove leading and trailing spaces
    F.trim("name").alias("name"),
)

silver_orders = (
    bronze_orders
    # drop orders without an amount
    .where(F.col("amount").isNotNull())
    .select(
        "order_id",
        "customer_id",
        # number with 2 decimals
        F.col("amount").cast("decimal(10,2)").alias("amount"),
    )
)

silver_orders.show()

## Gold
Revenue per customer, 0 if no orders.

In [ ]:
customer_revenue = (
    # left join keeps customers without orders
    silver_customers.join(silver_orders, "customer_id", "left")
    .groupBy("customer_id", "name")
    .agg(F.coalesce(F.sum("amount"), F.lit(0)).alias("revenue"))
    .orderBy("customer_id")
)

customer_revenue.show()

## Quality checks
One row per check on the raw source tables.

In [ ]:
c = bronze_customers
o = bronze_orders

# all customer checks in one pass over the table
cs = c.agg(
    F.count("*").alias("total"),
    F.sum(F.col("name").isNull().cast("int")).alias("missing_name"),
    (F.count("*") - F.countDistinct("customer_id")).alias("dup_id"),
    F.sum((F.length("name") > 100).cast("int")).alias("long_name"),
).first()

# all order checks in one pass over the table
os_ = o.agg(
    F.count("*").alias("total"),
    F.sum(F.col("amount").isNull().cast("int")).alias("missing_amount"),
    (F.count("*") - F.countDistinct("order_id")).alias("dup_id"),
    F.sum(((F.col("amount") <= 0) | (F.col("amount") > 1_000_000_000)).cast("int")).alias("out_of_range"),
).first()

# table, check, value, min, max, severity
checks = [
    ("customers", "total rows", cs["total"], 1, 100, "fail"),
    ("customers", "missing name", cs["missing_name"], 0, 0, "warn"),
    ("customers", "duplicate customer_id", cs["dup_id"], 0, 0, "fail"),
    ("customers", "name too long", cs["long_name"], 0, 0, "warn"),
    ("orders", "total rows", os_["total"], 1, 100, "fail"),
    ("orders", "missing amount", os_["missing_amount"], 0, 0, "warn"),
    ("orders", "duplicate order_id", os_["dup_id"], 0, 0, "fail"),
    ("orders", "amount out of range", os_["out_of_range"], 0, 0, "fail"),
]

data_quality = (
    spark.createDataFrame(checks, "table_name string, check_name string, value long, min_value int, max_value int, severity string")
    # sum() over an empty table returns null, count it as 0
    .fillna(0, subset=["value"])
    # inside the range = pass, outside = the check's severity
    .withColumn(
        "status",
        F.when(F.col("value").between(F.col("min_value"), F.col("max_value")), "pass").otherwise(F.col("severity")),
    )
    .withColumn("checked_at", F.current_timestamp())
)

data_quality.show(truncate=False)